In [1]:
import sys

sys.path.append("../src")

import numpy as np
import pandas as pd
from utils import fix_seed, gerar_dados_sinteticos_queimadas

fix_seed(42)

df = gerar_dados_sinteticos_queimadas(n_samples=1000)

# Criar variáveis binárias baseadas em limiares operacionais do INMET/INPE
df["alta_temp"] = (df["temperatura_max"] >= 32).astype(int)  # Evento A
df["baixa_umid"] = (df["umidade_relativa"] <= 45).astype(int)  # Evento B
df["longa_estiagem"] = (df["dias_sem_chuva"] >= 10).astype(int)  # Evento C

print("=== VARIÁVEIS BINÁRIAS CRIADAS PARA A ETAPA PROBABILÍSTICA ===")
print(
    f"Proporção Alta Temp (A=1): {df['alta_temp'].mean()*100:.1f}%"
)
print(
    f"Proporção Baixa Umid (B=1): {df['baixa_umid'].mean()*100:.1f}%"
)
print(
    f"Proporção Longa Estiagem (C=1): {df['longa_estiagem'].mean()*100:.1f}%"
)
print(
    f"Proporção Risco Crítico (R=1): {df['risco_critico'].mean()*100:.1f}%"
)

=== VARIÁVEIS BINÁRIAS CRIADAS PARA A ETAPA PROBABILÍSTICA ===
Proporção Alta Temp (A=1): 59.6%
Proporção Baixa Umid (B=1): 49.0%
Proporção Longa Estiagem (C=1): 74.8%
Proporção Risco Crítico (R=1): 0.2%


In [2]:
# Tabela de contagem cruzada para A (alta_temp) e B (baixa_umid)
tabela_contagem = pd.crosstab(
    df["alta_temp"], df["baixa_umid"], margins=True, margins_name="Total"
)
tabela_conjunta = pd.crosstab(
    df["alta_temp"], df["baixa_umid"], normalize=True, margins=True, margins_name="Total"
)

print("=== TABELA DE DISTRIBUIÇÃO CONJUNTA P(Alta_Temp, Baixa_Umid) ===")
print(tabela_conjunta.round(4))

# Verificação da soma dos axiomas da probabilidade
soma_axioma = tabela_conjunta.loc[0, 0] + tabela_conjunta.loc[0, 1] + tabela_conjunta.loc[1, 0] + tabela_conjunta.loc[1, 1]
print(f"\nSoma dos 4 eventos atômicos: {soma_axioma:.4f} (Deve ser exatamente 1.0)")

=== TABELA DE DISTRIBUIÇÃO CONJUNTA P(Alta_Temp, Baixa_Umid) ===
baixa_umid      0      1  Total
alta_temp                      
0           0.209  0.195  0.404
1           0.301  0.295  0.596
Total       0.510  0.490  1.000

Soma dos 4 eventos atômicos: 1.0000 (Deve ser exatamente 1.0)


In [3]:
P_A1 = tabela_conjunta.loc[1, "Total"]  # P(Alta Temp = 1)
P_B1 = tabela_conjunta.loc["Total", 1]  # P(Baixa Umid = 1)
P_A1_and_B1 = tabela_conjunta.loc[1, 1]  # P(Alta Temp = 1, Baixa Umid = 1)

# P(A|B) = P(A, B) / P(B)
P_A1_given_B1 = P_A1_and_B1 / P_B1
P_B1_given_A1 = P_A1_and_B1 / P_A1

print("=== CÁLCULOS MARGINAIS E CONDICIONAIS ===")
print(f"P(Alta Temp = 1)                = {P_A1:.4f}")
print(f"P(Baixa Umid = 1)               = {P_B1:.4f}")
print(f"P(Alta Temp = 1, Baixa Umid = 1) = {P_A1_and_B1:.4f}")
print(f"P(Alta Temp = 1 | Baixa Umid = 1) = {P_A1_given_B1:.4f}")
print(f"P(Baixa Umid = 1 | Alta Temp = 1) = {P_B1_given_A1:.4f}")

=== CÁLCULOS MARGINAIS E CONDICIONAIS ===
P(Alta Temp = 1)                = 0.5960
P(Baixa Umid = 1)               = 0.4900
P(Alta Temp = 1, Baixa Umid = 1) = 0.2950
P(Alta Temp = 1 | Baixa Umid = 1) = 0.6020
P(Baixa Umid = 1 | Alta Temp = 1) = 0.4950


In [4]:
# P(A, B, C) = P(A) * P(B | A) * P(C | A, B)

# Frequência relativa observada diretamente na base
P_ABC_direto = np.mean((df["alta_temp"] == 1) & (df["baixa_umid"] == 1) & (df["longa_estiagem"] == 1))

# Decomposição via Regra da Cadeia
P_A = np.mean(df["alta_temp"] == 1)
P_B_given_A = np.mean((df["alta_temp"] == 1) & (df["baixa_umid"] == 1)) / P_A

sub_AB = df[(df["alta_temp"] == 1) & (df["baixa_umid"] == 1)]
P_C_given_AB = np.mean(sub_AB["longa_estiagem"] == 1)

P_ABC_cadeia = P_A * P_B_given_A * P_C_given_AB

print("=== DEMONSTRAÇÃO DA REGRA DA CADEIA ===")
print(f"P(A=1)                  = {P_A:.4f}")
print(f"P(B=1 | A=1)            = {P_B_given_A:.4f}")
print(f"P(C=1 | A=1, B=1)       = {P_C_given_AB:.4f}")
print(f"P(A, B, C) via Cadeia   = {P_ABC_cadeia:.4f}")
print(f"P(A, B, C) Medição Direta = {P_ABC_direto:.4f}")

=== DEMONSTRAÇÃO DA REGRA DA CADEIA ===
P(A=1)                  = 0.5960
P(B=1 | A=1)            = 0.4950
P(C=1 | A=1, B=1)       = 0.9525
P(A, B, C) via Cadeia   = 0.2810
P(A, B, C) Medição Direta = 0.2810


In [5]:
# Problema Causal vs Diagnóstico:
# Queremos calcular P(Risco Crítico = 1 | Baixa Umid = 1) usando o Teorema de Bayes
# P(R=1 | B=1) = [ P(B=1 | R=1) * P(R=1) ] / P(B=1)

P_R1 = np.mean(df["risco_critico"] == 1)
P_B1_given_R1 = np.mean(df[df["risco_critico"] == 1]["baixa_umid"] == 1)
P_B1_total = np.mean(df["baixa_umid"] == 1)

P_R1_given_B1_bayes = (P_B1_given_R1 * P_R1) / P_B1_total

# Validação por contagem direta
P_R1_given_B1_direto = np.mean(df[df["baixa_umid"] == 1]["risco_critico"] == 1)

print("=== TEOREMA DE BAYES ===")
print(f"Prior P(Risco Crítico = 1)               = {P_R1:.4f}")
print(f"Verossimilhança P(Baixa Umid=1 | Risco=1) = {P_B1_given_R1:.4f}")
print(f"Evidência P(Baixa Umid = 1)               = {P_B1_total:.4f}")
print(f"Posterior via Bayes P(Risco=1 | Baixa Umid=1) = {P_R1_given_B1_bayes:.4f}")
print(f"Posterior via Contagem Direta             = {P_R1_given_B1_direto:.4f}")

=== TEOREMA DE BAYES ===
Prior P(Risco Crítico = 1)               = 0.0020
Verossimilhança P(Baixa Umid=1 | Risco=1) = 1.0000
Evidência P(Baixa Umid = 1)               = 0.4900
Posterior via Bayes P(Risco=1 | Baixa Umid=1) = 0.0041
Posterior via Contagem Direta             = 0.0041


In [6]:
# Verificar se P(Alta Temp, Baixa Umid | Risco Crítico) = P(Alta Temp | Risco) * P(Baixa Umid | Risco)
df_risco = df[df["risco_critico"] == 1]

P_A_and_B_given_R = np.mean((df_risco["alta_temp"] == 1) & (df_risco["baixa_umid"] == 1))
P_A_given_R = np.mean(df_risco["alta_temp"] == 1)
P_B_given_R = np.mean(df_risco["baixa_umid"] == 1)

P_indep_produto = P_A_given_R * P_B_given_R

print("=== TESTE DE INDEPENDÊNCIA CONDICIONAL (Dado Risco Crítico) ===")
print(f"P(Alta Temp, Baixa Umid | R=1) conjunta : {P_A_and_B_given_R:.4f}")
print(f"P(Alta Temp | R=1) * P(Baixa Umid | R=1): {P_indep_produto:.4f}")
print(f"Diferença Absoluta: {abs(P_A_and_B_given_R - P_indep_produto):.4f}")

=== TESTE DE INDEPENDÊNCIA CONDICIONAL (Dado Risco Crítico) ===
P(Alta Temp, Baixa Umid | R=1) conjunta : 0.5000
P(Alta Temp | R=1) * P(Baixa Umid | R=1): 0.5000
Diferença Absoluta: 0.0000
